# 04 · Serve on vLLM: one endpoint, several adapters

**What you will do**
1. Deploy Granite 4.1-3B on vLLM through OpenShift AI, with every trained adapter on the **same endpoint**.
2. Compare the untuned model and the adapters on held-out data, through the API.
3. Add an adapter to the running server without a restart.

**GPU:** one, for the server (about 10 GB). **Time:** 10–15 minutes.

| | Dev environment | On OpenShift AI |
|---|---|---|
| Serving | Run and secure `vllm serve` yourself | Managed vLLM endpoint (KServe), with TLS and a GPU size from a menu |
| Adapters | Often one model per process | Base + several LoRA adapters on one endpoint |
| Visibility | Your own scripts | The endpoint appears in the dashboard (Deploy > Deployments) |

In [ ]:
# Settings used by every notebook. Change them here if your project differs.
import os, json, time
from pathlib import Path

NAMESPACE  = open("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read().strip()   # this project
WS         = Path("/opt/app-root/src/workspace")   # the project volume, as mounted in this workbench
PVC        = "workspace"                           # its name; training and serving pods mount it at /workspace
REPO       = Path.cwd().parent                     # this repository (data/, sdg_flows/)
BASE_MODEL = "ibm-granite/granite-4.1-3b"
EXPERIMENT = "sentinel7"                           # MLflow experiment
NODE_SELECTOR = {}   # only with ReadWriteOnce storage: keep every pod that mounts the volume on one node
TRAINING_IMAGE = ("registry.redhat.io/rhoai/odh-th-torch-cuda-py312-rhel9@sha256:"   # OpenShift AI 3.5 Training Hub image
                  "884aeeb039f9592252fe0c518acf31803a208757d4fa6500fe79884a022ea52d")

os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", EXPERIMENT)
print("project:", NAMESPACE, "| volume:", WS, "| MLflow:", os.environ.get("MLFLOW_TRACKING_URI", "not enabled"))

In [ ]:
# Create OpenShift AI resources with the workbench's own identity (no token to paste).
import yaml
from kubernetes import client as k8s, config, dynamic
config.load_incluster_config()
dyn = dynamic.DynamicClient(k8s.ApiClient())

def apply(manifest: str):
    """Create or update every object in a YAML string, in this project."""
    for obj in yaml.safe_load_all(manifest):
        res = dyn.resources.get(api_version=obj["apiVersion"], kind=obj["kind"])
        name = obj["metadata"]["name"]
        try:
            dyn.create(res, body=obj, namespace=NAMESPACE); print("created", obj["kind"], name)
        except k8s.ApiException as e:
            if e.status != 409: raise
            dyn.patch(res, body=obj, name=name, namespace=NAMESPACE, content_type="application/merge-patch+json")
            print("updated", obj["kind"], name)

def run_job(manifest: str, show=None):
    """(Re)start a Kubernetes Job and stream its log. `show` keeps only lines containing one of these words."""
    obj = yaml.safe_load(manifest); name = obj["metadata"]["name"]
    batch, core = k8s.BatchV1Api(), k8s.CoreV1Api()
    try:
        batch.delete_namespaced_job(name, NAMESPACE, propagation_policy="Foreground")
        while batch.list_namespaced_job(NAMESPACE, field_selector=f"metadata.name={name}").items: time.sleep(3)
    except k8s.ApiException: pass
    if NODE_SELECTOR: obj["spec"]["template"]["spec"]["nodeSelector"] = NODE_SELECTOR
    batch.create_namespaced_job(NAMESPACE, obj)
    while True:
        pods = core.list_namespaced_pod(NAMESPACE, label_selector=f"job-name={name}").items
        if pods and pods[0].status.phase in ("Running", "Succeeded", "Failed"): break
        time.sleep(5)
    for raw in core.read_namespaced_pod_log(pods[0].metadata.name, NAMESPACE, follow=True, _preload_content=False):
        for line in raw.decode(errors="replace").splitlines():
            if show is None or any(w in line for w in show): print(line, flush=True)
    s = batch.read_namespaced_job(name, NAMESPACE).status
    print(f"\nJob {name}: {'succeeded' if s.succeeded else 'FAILED - see the log above'}")

def wait_ready(kind, api_version, name, timeout=1800):
    res = dyn.resources.get(api_version=api_version, kind=kind); t0 = time.time()
    while time.time() - t0 < timeout:
        conds = (dyn.get(res, name=name, namespace=NAMESPACE).to_dict().get("status") or {}).get("conditions", [])
        if any(c["type"] == "Ready" and c["status"] == "True" for c in conds):
            return print(f"{name} is ready ({time.time() - t0:.0f} s)")
        time.sleep(10)
    raise TimeoutError(name)

In [ ]:
import sys; sys.path.insert(0, str(WS / "code"))     # report_check.py and burst_rules.py, written by notebook 01
from report_check import check, scorecard as report_scorecard
from burst_rules import check as burst_check, scorecard as burst_scorecard

## 1 · Put the adapters where the server can read them

The server mounts the base model's folder, so adapters go in a `lora/` folder inside it.

In [ ]:
import shutil
lora_dir = WS / "models/granite-4.1-3b/lora"; lora_dir.mkdir(exist_ok=True)
for name in ["edgecopilot-sft", "edgecopilot-sft-v2", "burst-sft"]:
    src = WS / "checkpoints" / name
    if (src / "adapter_model.safetensors").exists():
        shutil.copytree(src, lora_dir / name, dirs_exist_ok=True, ignore=shutil.ignore_patterns("_data", "checkpoint-*"))
ADAPTERS = sorted(p.name for p in lora_dir.iterdir() if (p / "adapter_model.safetensors").exists())
print("adapters:", ADAPTERS)

## 2 · Deploy

One resource, an `LLMInferenceService`, describes the whole deployment. It starts from the vLLM template that ships
with OpenShift AI 3.5.

In [ ]:
NAME = "granite-41-3b"
lora_args = "".join(f"\n      - {a}=/mnt/models/lora/{a}" for a in ADAPTERS)
SERVE = f"""
apiVersion: serving.kserve.io/v1alpha1
kind: LLMInferenceService
metadata:
  name: {NAME}
  annotations: {{openshift.io/display-name: "Sentinel-7 Granite 4.1-3B + adapters"}}
  labels: {{opendatahub.io/dashboard: "true"}}
spec:
  baseRefs: [{{name: v3-5-0-kserve-config-llm-single-node-template-nvidia-cuda}}]
  model: {{name: granite-4.1-3b, uri: "pvc://{PVC}/models/granite-4.1-3b"}}
  replicas: 1
  template:
    {"nodeSelector: " + json.dumps(NODE_SELECTOR) if NODE_SELECTOR else ""}
    containers:
    - name: main
      args:
      - --max-model-len=8192
      - --gpu-memory-utilization=0.42      # about 10 GB on a 24 GB L4; raise it on a dedicated GPU
      - --max-num-seqs=8
      - --enable-lora
      - --max-loras=3
      - --max-lora-rank=16
      - --lora-modules{lora_args}
      env: [{{name: VLLM_ALLOW_RUNTIME_LORA_UPDATING, value: "True"}}]
      resources:
        requests: {{cpu: "1", memory: 12Gi, nvidia.com/gpu: "1"}}
        limits: {{cpu: "4", memory: 20Gi, nvidia.com/gpu: "1"}}
"""
apply(SERVE)
wait_ready("LLMInferenceService", "serving.kserve.io/v1alpha1", NAME)
URL = f"https://{NAME}-kserve-workload-svc.{NAMESPACE}.svc:8000/v1"
print(URL)

## 3 · Talk to it

In [ ]:
import httpx
from openai import OpenAI
llm = OpenAI(base_url=URL, api_key="none", http_client=httpx.Client(verify=False))   # in-cluster certificate
print([m.id for m in llm.models.list().data])

note = json.loads((WS / "data/edgecopilot/test.jsonl").open().readline())["messages"]
for model in ["granite-4.1-3b", "edgecopilot-sft"]:
    r = llm.chat.completions.create(model=model, messages=note[:2], temperature=0, max_tokens=700)
    print(f"\n{model}:\n{r.choices[0].message.content[:600]}")

## 4 · Before vs after, on held-out data

In [ ]:
from concurrent.futures import ThreadPoolExecutor
def ask(model, messages, max_tokens=700):
    return llm.chat.completions.create(model=model, messages=messages, temperature=0, max_tokens=max_tokens).choices[0].message.content

N = 50
test = [json.loads(l) for l in (WS / "data/edgecopilot/test.jsonl").open()][:N]
golds = [json.loads(r["messages"][2]["content"]) for r in test]
cards = {}
for model in ["granite-4.1-3b"] + [a for a in ADAPTERS if a.startswith("edgecopilot")]:
    with ThreadPoolExecutor(4) as ex:
        outs = list(ex.map(lambda r: ask(model, r["messages"][:2]), test))
    cards[model] = report_scorecard(outs, golds)
import pandas as pd
pd.DataFrame(cards).T

In [ ]:
burst_test = [json.loads(l) for l in (WS / "data/burst/test.jsonl").open()][:N]
tasks = [json.loads(r["ground_truth"]) for r in burst_test]
bcards = {}
for model in ["granite-4.1-3b"] + [a for a in ADAPTERS if a.startswith("burst")]:
    with ThreadPoolExecutor(4) as ex:
        outs = list(ex.map(lambda r: ask(model, r["messages"], 200), burst_test))
    bcards[model] = burst_scorecard(outs, tasks)
pd.DataFrame(bcards).T

The GRPO model was scored inside its training Job (notebook 03, `results/burst-grpo.json`). To serve it, merge its
adapter into `burst-sft-merged` and deploy that as a second model the same way.

## 5 · Add an adapter without a restart

In [ ]:
import requests
NEW = "edgecopilot-sft-v2"
if (lora_dir / NEW).exists():
    r = requests.post(URL.replace("/v1", "/v1/load_lora_adapter"), verify=False,
                      json={"lora_name": NEW + "-live", "lora_path": f"/mnt/models/lora/{NEW}"})
    print(r.status_code, r.text[:200]); print([m.id for m in llm.models.list().data])

## What we measured

| 300 held-out notes | Exact reports | Invented a field |
|---|---|---|
| Granite 4.1-3B untuned, 1,449-token prompt | 6.3% | 15% |
| Llama 3.3 70B / gpt-oss-120b (hosted), same prompt | 46.3% / 75.0% | about 2% |
| `edgecopilot-sft`, 94-token prompt | 92.7% | 7.0% |
| `edgecopilot-sft-v2` | 99.7–100% | 0% |

All three adapters and the base model ran on one endpoint in about 10 GB of GPU memory. **Next:** notebook 05 takes
the model to a laptop.